# 02 · Verificación de la reparación de datos

**Objetivo:** comprobar que el dataset de 68 variables quedó reparado y conserva
compatibilidad con los modelos anteriores. Este notebook es de lectura y no sobrescribe datos.

La reparación encontró el texto `Directo` en `monto_ult_vs_media`, fila 919
(posición desde cero). El valor correcto se reconstruyó desde el preprocesado;
se conservó un respaldo y un manifiesto con hashes en `reports/data_repair/`.

Orden de lectura: entradas → tipos → variables derivadas → comparación → métricas previas.
El nuevo experimento temporal se estudia en `05_modelling/06_optuna_validacion_temporal.ipynb`.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Abrir este notebook dentro del repositorio")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PROC = ROOT / "data/processed"

## 1. Archivos de entrada

`churn_dataset_processed.csv` es la etapa previa. `selected_features.json` conserva
el orden de las 68 variables del experimento anterior. La reparación no cambió esa selección.

In [2]:
original = pd.read_csv(PROC / "churn_dataset.csv", parse_dates=["mes_obs"])
previo = pd.read_csv(PROC / "churn_dataset_processed.csv", parse_dates=["mes_obs"])
reparado = pd.read_csv(PROC / "churn_dataset_features.csv", parse_dates=["mes_obs"])
seleccion = json.loads((PROC / "selected_features.json").read_text())
claves = ["id_vendedor", "mes_obs", "mes_rank", "churn"]
assert original[claves].equals(previo[claves])
assert original[claves].equals(reparado[claves])
assert not reparado.duplicated(["id_vendedor", "mes_rank"]).any()
pd.DataFrame({"etapa": ["original", "preprocesado", "reparado"],
              "filas": [len(original), len(previo), len(reparado)],
              "columnas": [original.shape[1], previo.shape[1], reparado.shape[1]]})

,etapa,filas,columnas
0,original,30356,46
1,preprocesado,30356,82
2,reparado,30356,72


## 2. Ningún texto, nulo o infinito puede entrar como variable numérica

In [3]:
X = reparado[seleccion]
assert X.select_dtypes(exclude="number").empty
assert np.isfinite(X.to_numpy(dtype=float)).all()
pd.Series({"variables": len(seleccion), "nulos": int(X.isna().sum().sum()),
           "columnas_texto": len(X.select_dtypes("object").columns),
           "valor_recuperado_fila_919": reparado.loc[919, "monto_ult_vs_media"]})

variables                    68.000000
nulos                         0.000000
columnas_texto                0.000000
valor_recuperado_fila_919     3.261011
dtype: float64

## 3. Reconstrucción independiente de las variables derivadas

Cada razón utiliza información de la misma fila. No se aprende nada del test.
Si el denominador es cero, se utiliza cero, igual que en la etapa original.

In [4]:
def engineer(df):
    d = df.copy()

    def ratio(a, b):
        return (a / b.replace(0, np.nan)).fillna(0)

    d["ticket_prom_u12"] = ratio(d.monto_u12, d.n_ped_u12)
    d["ticket_prom_u3"] = ratio(d.monto_u3, d.n_ped_u3)
    d["intensidad_u3"] = ratio(d.n_ped_u3, d.meses_activos_u3)
    d["basket_size_u12"] = ratio(d.n_prod_u12, d.n_ped_u12)
    d["recencia_norm"] = d.meses_desde_compra_previa * d.meses_activos_u12 / 12
    d["tasa_act_reciente_vs_hist"] = ratio(d.meses_activos_u3 / 3, d.meses_activos_u12 / 12)
    return d

In [5]:
esperado = engineer(previo)[claves + seleccion]
pd.testing.assert_frame_equal(esperado, reparado, check_dtype=False, rtol=1e-9, atol=1e-9)
manifiestos = sorted((ROOT / "reports/data_repair").glob("repair_*.json"))
evidencia = json.loads(manifiestos[-1].read_text())
print("Reconstrucción coincidente; diferencias corregidas:")
print(evidencia["changed_positions_zero_based"])
print("SHA256 del CSV reparado:", evidencia["after_sha256"])

Reconstrucción coincidente; diferencias corregidas:
{'monto_ult_vs_media': [919]}
SHA256 del CSV reparado: f178fbceffac23039fc46e45872ad3346a7300ed20577facbe4fc445262c30bb


## 4. Compatibilidad con los modelos anteriores

Se reproduce su inferencia en el OOT **ya consultado**. Esta tabla comprueba la
reparación; no se utiliza para optimizar el nuevo experimento temporal.

In [6]:
import joblib
from sklearn.metrics import roc_auc_score

test = reparado.mes_rank >= reparado.mes_rank.max() - 3
filas = []
for nombre in ["logreg", "rf", "xgboost", "catboost"]:
    modelo = joblib.load(ROOT / "models" / f"{nombre}_tuned.joblib")
    prob = modelo.predict_proba(reparado.loc[test, seleccion])[:, 1]
    auc = roc_auc_score(reparado.loc[test, "churn"], prob)
    registrado = json.loads((ROOT / "05_modelling" / f"{nombre}_best_params.json").read_text())["oot_auc"]
    assert np.isclose(auc, registrado, atol=1e-12)
    filas.append({"modelo": nombre, "AUC_OOT_reproducido": auc,
                  "diferencia_vs_registrado": auc - registrado})
pd.DataFrame(filas).set_index("modelo")

,AUC_OOT_reproducido,diferencia_vs_registrado
modelo,,
logreg,0.769260,0.0
rf,0.758982,0.0
xgboost,0.764820,0.0
catboost,0.765237,0.0


La reparación está verificada si todas las aserciones pasan. El código que realizó
la escritura y el respaldo está en `scripts/temporal_optuna.py`, función `repair`.
No es necesario repetir la reparación para entrenar: el nuevo flujo parte del dataset original.